[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/LogReg_Titanic_Pair.ipynb)

# Titanic Odds Ratios: What Does a Coefficient Mean?
**CS 377 · companion to 18_2_0_1 and 18_2_1 Logistic Regression with Titanic · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Titanic Odds Ratios quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Some sections have a hint about the question itself (the change to make is always given): try without it first.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1.

It repeats what 18_2_1 does before its "Interpreting the Model" section: it loads the Titanic data, builds the same seven features (`sex` is 0 for male and 1 for female; `pclass_2` and `pclass_3` are compared with 1st class), makes the same 75/25 split and fits the same pipeline. It makes what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: 665 training and 222 test passengers; in `y`, 1 means the passenger survived and 0 means they died.
- `df`: the full data, including each passenger's `name`.
- `pipeline`: the fitted 18_2_1 model (`StandardScaler` on `age` and `fare_log`, then `LogisticRegression`).
- `y_proba`: the model's P(survive) for each test passenger.
- `feature_names`: the seven features in the order of the model's coefficients.
- `make_model(class_weight=None, C=1.0)`: the same pipeline, not yet fitted.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression

pd.set_option('display.width', 140)

# 18_2_1: load the data and build the same seven features
df = pd.read_csv('https://raw.githubusercontent.com/bsheese/cs377/main/data/data_titanic.csv')
df.columns = df.columns.str.lower().str.replace(' aboard', '', regex=False)
df['sibspouse'] = (df['siblings/spouses'] > 0).astype(int)
df['parentchild'] = (df['parents/children'] > 0).astype(int)
df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df['fare_log'] = np.log1p(df['fare'])
df = pd.get_dummies(df, columns=['pclass'], prefix='pclass', drop_first=True)   # 1st class is the reference

features = ['age', 'sibspouse', 'parentchild', 'fare_log', 'sex', 'pclass_2', 'pclass_3']
X = df[features]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


def make_model(class_weight=None, C=1.0):
    """The 18_2_1 pipeline, not yet fitted: scale age and fare_log, pass the 0/1 columns through, then
    logistic regression. class_weight and C are left at their defaults unless a section changes them."""
    preprocessor = ColumnTransformer([('num', StandardScaler(), ['age', 'fare_log'])], remainder='passthrough')
    return Pipeline([('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(random_state=42, class_weight=class_weight, C=C))])


pipeline = make_model().fit(X_train, y_train)            # the 18_2_1 model
y_proba = pipeline.predict_proba(X_test)[:, 1]           # P(survive) for the 222 test passengers
feature_names = [n.split('__')[-1] for n in pipeline.named_steps['preprocessor'].get_feature_names_out()]

print(f"training passengers: {len(y_train)} ({y_train.mean():.1%} survived)   "
      f"test passengers: {len(y_test)} ({y_test.mean():.1%} survived)")
print(f"test accuracy: {pipeline.score(X_test, y_test):.3f}")

# Part 2 (Q3-Q6): The 18_2_1 Model in the Notebook

## Q3 · The tie with the sex-only rule
> **Quiz Q3:** On the 18_2_1 test set, the 7-feature model and the rule "women survive, men die" have the same accuracy, 0.770. One test set can produce a tie by luck, so the Q3 cell compares the two with 5-fold cross-validation on the training set. The rule is fitted as a logistic regression that uses only the `sex` column. Right now the cell scores accuracy only. Add `'roc_auc'` to `SCORING` and run it.
>
> - What is each model's mean CV accuracy?
> - What is each model's mean CV AUC?
> - The cell prints the only probabilities the sex-only model ever gives. Use them to explain why its AUC is lower.

**✏️ Your change:** Change `SCORING = ['accuracy']` to `SCORING = ['accuracy', 'roc_auc']`, then run the cell.

**Tell your Recorder:** each model's mean CV accuracy, each model's mean CV AUC, then why the sex-only model's AUC is lower.

<details><summary>Hint</summary>

AUC takes one survivor and one non-survivor and asks whether the survivor got the higher probability. If both are men, what probabilities does the sex-only model give them?

</details>

In [ ]:
SCORING = ['accuracy']      # ✏️ add 'roc_auc' to this list

sex_only = LogisticRegression(random_state=42)   # fitted on the sex column alone

rows = []
for scoring in SCORING:
    for name, model, X_cols in [('7 features', make_model(), X_train), ('sex only', sex_only, X_train[['sex']])]:
        scores = cross_val_score(model, X_cols, y_train, cv=5, scoring=scoring)
        rows.append({'scoring': scoring, 'model': name,
                     **{f'fold {i}': s for i, s in enumerate(scores, 1)}, 'mean': scores.mean()})

sex_only.fit(X_train[['sex']], y_train)
print('P(survive) values the sex-only model gives:',
      np.unique(sex_only.predict_proba(X_test[['sex']])[:, 1]).round(3), '(men, women)')
pd.DataFrame(rows).set_index(['scoring', 'model']).round(3)

## Q4 · Coefficients as odds ratios
> **Quiz Q4:** 18_2_1 turns each coefficient into an odds ratio. The Q4 cell shows the coefficients (`beta`), but the `odds_ratio` column is empty. Replace `np.nan` so the column holds the odds ratios, and run it.
>
> - Write one sentence about the odds ratio for `sex` that uses the word "odds".
> - By what percent are a 3rd-class passenger's odds of survival lower than a 1st-class passenger's, all else equal?
> - `age` was standardized. What is one unit of `age` in this table, and what does the cell say happens to the odds for a passenger 10 years older, all else equal?

**✏️ Your change:** Replace `np.nan` with `np.exp(betas['beta'])`.

**Tell your Recorder:** the sentence about sex, the percent for 3rd class, then what one unit of age is and what 10 years older does to the odds.

<details><summary>Hint 1</summary>

An odds ratio of 0.168 means the odds are multiplied by 0.168. If something is multiplied by 0.168, by what percent did it go down?

</details>

<details><summary>Hint 2</summary>

An odds ratio describes odds, not probability, so a sentence that says "12 times as likely to survive" says something different from what the table shows.

</details>

In [ ]:
betas = pd.DataFrame({'feature': feature_names,
                      'beta': pipeline.named_steps['classifier'].coef_[0]})
betas['odds_ratio'] = np.nan      # ✏️ turn each beta into an odds ratio
print(betas.sort_values('odds_ratio', ascending=False).round(3).to_string(index=False))

age_sd = pipeline.named_steps['preprocessor'].named_transformers_['num'].scale_[0]
age_beta = betas.set_index('feature').loc['age', 'beta']
print(f"\nOne unit of age in this table is one standard deviation of age: {age_sd:.1f} years")
print(f"Odds ratio for 10 years older: exp(beta * 10 / {age_sd:.1f}) = {np.exp(age_beta * 10 / age_sd):.3f}")

## Q5 · Same odds ratio, different change in probability
> **Quiz Q5:** The Q5 cell takes five men from the test set and makes a copy of each one with `sex` changed and everything else the same. Right now the copies still have `sex = 0`. Change it to `1` and run it.
>
> - What is the odds ratio in every row?
> - In Q2 the man who started at 0.02 gained less than the man who started at 0.50. Does the man here with the lowest P(survive) as male also gain the least? Does the man who starts closest to 0.50 gain the most? Give the largest and smallest gains in percentage points.
> - Someone asks: "How much does being female raise a passenger's chance of survival in this model?" Write an answer that is true for all five men.

**✏️ Your change:** Change `as_female['sex'] = 0` to `as_female['sex'] = 1`, then run the cell.

**Tell your Recorder:** the odds ratio in each row, yes or no to each of the two checks with the largest and smallest gains, then your answer to the question.

<details><summary>Hint</summary>

For the last bullet: any single number of percentage points will be wrong for some of the five men. Which number in the table is the same for all five?

</details>

In [ ]:
men = X_test.loc[[324, 745, 180, 427, 138]]     # five men from the test set
as_female = men.copy()
as_female['sex'] = 0      # ✏️ make the copies female (1)

p_male = pipeline.predict_proba(men)[:, 1]
p_female = pipeline.predict_proba(as_female)[:, 1]

pd.DataFrame({
    'name': df.loc[men.index, 'name'],
    'class': 1 + men['pclass_2'].astype(int) + 2 * men['pclass_3'].astype(int),
    'age': men['age'],
    'P(survive) as male': p_male,
    'P(survive) as female': p_female,
    'odds ratio': (p_female / (1 - p_female)) / (p_male / (1 - p_male)),
    'change in P': p_female - p_male,
}).round(3)

## Q6 · Can you read the probabilities as percentages?
> **Quiz Q6:** 18_2_1 says logistic regression's probabilities tend to be well calibrated: among passengers it gives about 70%, about 70% survive. The Q6 cell checks this on out-of-fold probabilities for the 665 training passengers, grouped into bins, for two models. Right now both use `class_weight=None`. Change the second to `'balanced'` and run it. (`'balanced'` gives survivors more weight in training, the way `scale_pos_weight` did for XGBoost in 18_1.)
>
> - In the (0.2, 0.4] bin, what are the mean predicted probability and the actual survival rate for each model?
> - What is each model's overall mean P(survive)? The actual survival rate is 0.385.
> - Which model's probabilities could you use to say something like "this passenger has a 35% chance of surviving," and what in the table shows it?

**✏️ Your change:** Change `WEIGHTS = [None, None]` to `WEIGHTS = [None, 'balanced']`, then run the cell.

**Tell your Recorder:** the (0.2, 0.4] bin for each model, each model's mean P(survive) against 0.385, then which model's probabilities you could read as chances and what in the table shows it.

<details><summary>Hint</summary>

A probability can be read as a chance when, among the passengers given about that probability, about that share actually survived. Compare `mean_predicted` with `actual_rate` in the same row.

</details>

In [ ]:
WEIGHTS = [None, None]      # ✏️ change the second to 'balanced'

bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
tables = []
for w in WEIGHTS:
    oof = cross_val_predict(make_model(class_weight=w), X_train, y_train, cv=5,
                            method='predict_proba')[:, 1]     # out-of-fold P(survive)
    d = pd.DataFrame({'P(survive)': oof, 'survived': y_train.values})
    d['bin'] = pd.cut(d['P(survive)'], bins)
    t = d.groupby('bin', observed=True).agg(passengers=('survived', 'size'),
                                            mean_predicted=('P(survive)', 'mean'),
                                            actual_rate=('survived', 'mean'))
    t.insert(0, 'class_weight', str(w))
    tables.append(t)
    print(f'class_weight={str(w):9}  mean P(survive) {oof.mean():.3f}')

print(f'actual survival rate in the training set: {y_train.mean():.3f}')
pd.concat(tables).round(3)

# Extra credit (Q7-Q8): Thresholds and the Default Penalty
Only if you have finished Q1-Q6.

## ⭐ Q7 · A threshold on probability is a threshold on log-odds
> **Quiz Q7:** 18_2_1's threshold sweep found that 0.4 gave the highest test accuracy. The Q7 cell flags test passengers as survivors in two ways: P(survive) ≥ 0.4, and log-odds z ≥ `z_cut`, where `z` comes from `decision_function` (the log-odds before the sigmoid). Replace `np.nan` with the log-odds that matches a probability of 0.4, and run it.
>
> - What is `z_cut`, and do the two rules flag the same passengers?
> - A 0.5 threshold flags a passenger when z ≥ 0. If you kept the 0.5 threshold, what would the intercept (printed by the cell) have to be for the model to flag the same passengers as the 0.4 threshold does?

**✏️ Your change:** Replace `np.nan` with `np.log(THRESHOLD / (1 - THRESHOLD))`.

**Tell your Recorder:** z_cut, whether the two rules agree, then the intercept that would do the same job at 0.5.

<details><summary>Hint</summary>

Watch the sign. `z_cut` is negative, so a passenger with z = −0.3 is flagged at 0.4 but not at 0.5. To flag that passenger at 0.5, every z has to go up by the size of `z_cut`, and the intercept is the part of z that every passenger shares.

</details>

In [ ]:
THRESHOLD = 0.4
z = pipeline.decision_function(X_test)      # log-odds of survival for each test passenger
z_cut = np.nan      # ✏️ the log-odds that matches a probability of THRESHOLD

by_p = y_proba >= THRESHOLD
by_z = z >= z_cut
print(f'z_cut = {z_cut:.3f}')
print(f'flagged by P >= {THRESHOLD}: {by_p.sum()}   flagged by z >= z_cut: {by_z.sum()}   '
      f'same passengers: {(by_p == by_z).all()}')
print(f"intercept: {pipeline.named_steps['classifier'].intercept_[0]:.3f}")

## ⭐ Q8 · sklearn's default penalty
> **Quiz Q8:** scikit-learn's `LogisticRegression` adds a ridge (L2) penalty by default, like the Ridge regression in 17_2_1_3. `C` is the inverse of the penalty strength: the default is `C=1`, and a very large `C` means almost no penalty. The Q8 cell fits the model with two values of `C` and shows the odds ratios. Change the second to `1e6` and run it.
>
> - For most features, do the odds ratios move toward 1 or away from 1 when the penalty is removed? Give the numbers for `sex` and `pclass_3`.
> - One odds ratio moves the other way. Which one, and which other feature, closely related to it, might explain why?

**✏️ Your change:** Change `C_VALUES = [1.0, 1.0]` to `C_VALUES = [1.0, 1e6]`, then run the cell.

**Tell your Recorder:** the direction most odds ratios move with the numbers for sex and pclass_3, then the one that moves the other way and why.

<details><summary>Hint</summary>

Which passengers paid the highest fares? If two features carry much of the same information, think back to how Ridge split the credit between the twin features in 17_2_1_3.

</details>

In [ ]:
C_VALUES = [1.0, 1.0]      # ✏️ change the second to 1e6 (almost no penalty)

odds = {}
for C in C_VALUES:
    m = make_model(C=C).fit(X_train, y_train)
    odds[f'odds ratio, C={C:g}'] = np.exp(m.named_steps['classifier'].coef_[0])
pd.DataFrame(odds, index=feature_names).round(3)